In [1]:
import subprocess, sys, time, random, platform
from pathlib import Path
import numpy as np, pandas as pd, torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = "llm-hallucination"
if not Path(REPO).exists():
    subprocess.run(["git", "clone", f"https://github.com/Tonggi-Simanjuntak/{REPO}.git"], check=True)
else:
    print(subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True).stdout)

# ---- QA versi akhir (dengan pengaman) ----
qa = pd.read_csv(Path(REPO) / "data/processed/qa.csv", dtype=str, keep_default_na=False)
assert len(qa) == 94, f"jumlah QA {len(qa)}, seharusnya 94"
kosong_status = (qa.status_cek.str.strip() == "").sum()
kosong_pem = (qa.pemeriksa.str.strip() == "").sum()
assert kosong_status == 0 and kosong_pem == 0, (
    f"qa.csv BELUM versi akhir: status_cek kosong {kosong_status}, pemeriksa kosong {kosong_pem}. Push versi akhir dulu.")
print("QA:", len(qa), "| split:", qa.split.value_counts().to_dict(),
      "| status_cek:", qa.status_cek.value_counts().to_dict())

# ---- konfigurasi (dikunci setelah dev) ----
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
REVISI = "cdbee75f17c01a7cc42f958dc650907174af0554"
SEED = 42
MAX_NEW = 256
SYSTEM = "Anda adalah asisten layanan publik Indonesia. Jawab dalam bahasa Indonesia dengan ringkas."

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print("\nLingkungan:", torch.cuda.get_device_name(0), "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| python", platform.python_version())

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISI)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISI, dtype=torch.float16, device_map="cuda")
model.eval()
print(f"Model dimuat dalam {time.time()-t0:.1f} detik | memori GPU {torch.cuda.memory_allocated()/1e9:.2f} GB")

eos = model.generation_config.eos_token_id
EOS = set(eos if isinstance(eos, (list, tuple)) else [eos]) | {tokenizer.eos_token_id}

def jawab(pertanyaan):
    pesan = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": pertanyaan}]
    teks = tokenizer.apply_chat_template(pesan, tokenize=False, add_generation_prompt=True)
    masuk = tokenizer([teks], return_tensors="pt").to(model.device)
    torch.cuda.synchronize(); t = time.time()
    with torch.no_grad():
        out = model.generate(**masuk, max_new_tokens=MAX_NEW, do_sample=False,
                             temperature=None, top_p=None, top_k=None)
    torch.cuda.synchronize(); dt = time.time() - t
    gen = out[0][masuk["input_ids"].shape[1]:]
    n = len(gen)
    return dict(jawaban=tokenizer.decode(gen, skip_special_tokens=True).strip(),
                jumlah_token=n,
                terpotong=bool(n >= MAX_NEW and gen[-1].item() not in EOS),
                waktu_detik=round(dt, 2))

print("\nKONFIGURASI:", dict(model=MODEL_ID, revisi=REVISI, dtype="float16", decoding="greedy",
      max_new_tokens=MAX_NEW, seed=SEED, system_prompt=SYSTEM))

QA: 94 | split: {'uji': 63, 'dev': 31} | status_cek: {'ok': 94}

Lingkungan: Tesla T4 | torch 2.14.1+cu130 | transformers 5.22.1 | python 3.13.16


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model dimuat dalam 116.3 detik | memori GPU 8.04 GB

KONFIGURASI: {'model': 'Qwen/Qwen3-4B-Instruct-2507', 'revisi': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'dtype': 'float16', 'decoding': 'greedy', 'max_new_tokens': 256, 'seed': 42, 'system_prompt': 'Anda adalah asisten layanan publik Indonesia. Jawab dalam bahasa Indonesia dengan ringkas.'}


In [2]:
uji = [
    "Berapa biaya paspor biasa elektronik 48 halaman?",
    "Bagaimana cara mengganti paspor yang masa berlakunya hampir habis?",
    "Apa yang harus dilakukan jika Kartu Keluarga rusak?",
    "Apa yang dimaksud dengan SKPLN?",
    "Bagaimana prosedur pindah domisili ke kota lain bagi WNI?",
]
for i, p in enumerate(uji, 1):
    r = jawab(p)
    print(f"\n=== {i}. {p}")
    print(r["jawaban"])
    print(f"[token={r['jumlah_token']} | terpotong={r['terpotong']} | {r['waktu_detik']} detik]")
print(f"\nMemori GPU puncak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


=== 1. Berapa biaya paspor biasa elektronik 48 halaman?
Biaya paspor biasa elektronik 48 halaman di Indonesia saat ini adalah **Rp 175.000** (sebagai informasi hingga 2024). Harga ini bisa berubah tergantung kebijakan pemerintah, jadi disarankan untuk memeriksa langsung di situs resmi Kementerian Hukum dan HAM atau di kantor paspor terdekat.
[token=99 | terpotong=False | 7.41 detik]

=== 2. Bagaimana cara mengganti paspor yang masa berlakunya hampir habis?
Untuk mengganti paspor dengan masa berlaku hampir habis, ikuti langkah-langkah berikut:

1. **Periksa masa berlaku paspor** – Pastikan paspor Anda akan habis dalam 6 bulan ke depan (sesuai ketentuan pemerintah).

2. **Siapkan dokumen**:
   - Paspor lama (dengan halaman data)
   - KTP
   - Fotokopi KTP
   - 2 foto berwarna ukuran 3,5 x 4,5 cm (baru)
   - Surat keterangan dari instansi jika diperlukan (misalnya untuk pekerjaan atau studi)

3. **Buat aplikasi online** (opsional) – Bisa melalui situs resmi Kementerian Hukum dan HAM atau

In [7]:
import time
import pandas as pd
from pathlib import Path

OUT = Path(REPO) / "data" / "processed" / "jawaban_s1_dev.csv"
dev = qa[qa.split == "dev"].reset_index(drop=True)
assert len(dev) == 31, f"soal dev {len(dev)}, seharusnya 31"

hasil = pd.read_csv(OUT, dtype=str, keep_default_na=False).to_dict("records") if OUT.exists() else []
selesai = {r["id_qa"] for r in hasil}
print(f"Soal dev: {len(dev)} | sudah selesai: {len(selesai)}")

t_mulai = time.time()
for i, row in dev.iterrows():
    if row.id_qa in selesai:
        continue
    r = jawab(row.pertanyaan)          # hanya pertanyaan yang dikirim ke model
    hasil.append(dict(id_qa=row.id_qa, split=row.split, tipe=row.tipe, pertanyaan=row.pertanyaan,
                      jawaban_model=r["jawaban"], jumlah_token=r["jumlah_token"],
                      terpotong=r["terpotong"], waktu_detik=r["waktu_detik"],
                      max_new_tokens=MAX_NEW, seed=SEED))
    if len(hasil) % 5 == 0:
        pd.DataFrame(hasil).to_csv(OUT, index=False, encoding="utf-8")
        print(f"  {len(hasil)}/{len(dev)} selesai ({time.time()-t_mulai:.0f} detik)")
pd.DataFrame(hasil).to_csv(OUT, index=False, encoding="utf-8")

d = pd.DataFrame(hasil)
d["jumlah_token"] = d.jumlah_token.astype(int)
d["waktu_detik"] = d.waktu_detik.astype(float)
d["terpotong"] = d.terpotong.astype(str).str.lower().eq("true")

print("\n===== RINGKASAN DEV =====")
print(f"Jumlah jawaban      : {len(d)}")
print(f"Terpotong           : {d.terpotong.sum()} ({d.terpotong.mean()*100:.1f}%)  [target di bawah sekitar 5%]")
print(f"Token  rata/median/maks: {d.jumlah_token.mean():.0f} / {d.jumlah_token.median():.0f} / {d.jumlah_token.max()}")
print(f"Waktu  rata/maks (detik): {d.waktu_detik.mean():.1f} / {d.waktu_detik.max():.1f} | total {d.waktu_detik.sum()/60:.1f} menit")
print(f"Memori GPU puncak   : {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("\nTerpotong per tipe soal:")
print(d.groupby("tipe").terpotong.agg(["sum", "count"]).to_string())

print("\n--- Soal yang terpotong ---")
for _, x in d[d.terpotong].iterrows():
    print(f"{x.id_qa} [{x.tipe}] {x.pertanyaan}  (token={x.jumlah_token})")

print("\n--- 4 contoh jawaban pertama (maks 300 karakter) ---")
for _, x in d.head(4).iterrows():
    print(f"\n{x.id_qa} | {x.pertanyaan}\n> {x.jawaban_model[:300]}")

Soal dev: 31 | sudah selesai: 0
  5/31 selesai (75 detik)
  10/31 selesai (119 detik)
  15/31 selesai (164 detik)
  20/31 selesai (183 detik)
  25/31 selesai (226 detik)
  30/31 selesai (261 detik)

===== RINGKASAN DEV =====
Jumlah jawaban      : 31
Terpotong           : 0 (0.0%)  [target di bawah sekitar 5%]
Token  rata/median/maks: 161 / 138 / 472
Waktu  rata/maks (detik): 9.1 / 26.2 | total 4.7 menit
Memori GPU puncak   : 8.17 GB

Terpotong per tipe soal:
          sum  count
tipe                
Biaya       0      1
Durasi      0      5
Instansi    0      3
Prosedur    0      4
Syarat      0     18

--- Soal yang terpotong ---

--- 4 contoh jawaban pertama (maks 300 karakter) ---

QA001 | Kepada siapa permohonan paspor baru bagi anak berkewarganegaraan Indonesia yang lahir di luar wilayah Indonesia diajukan?
> Permohonan paspor baru bagi anak berkewarganegaraan Indonesia yang lahir di luar wilayah Indonesia diajukan kepada **Kementerian Kebudayaan, Kepemilikan, dan Kewarganegaraan 

In [8]:
import time
import pandas as pd
from pathlib import Path

MAX_NEW = 384   # dinaikkan dari 256 (terpotong 22,6% di dev); fungsi jawab() membaca nilai ini

OUT = Path(REPO) / "data" / "processed" / "jawaban_s1_dev_384.csv"
dev = qa[qa.split == "dev"].reset_index(drop=True)
assert len(dev) == 31, f"soal dev {len(dev)}, seharusnya 31"

hasil = pd.read_csv(OUT, dtype=str, keep_default_na=False).to_dict("records") if OUT.exists() else []
selesai = {r["id_qa"] for r in hasil}
print(f"Batas jawaban: {MAX_NEW} token | soal dev: {len(dev)} | sudah selesai: {len(selesai)}")

t_mulai = time.time()
for i, row in dev.iterrows():
    if row.id_qa in selesai:
        continue
    r = jawab(row.pertanyaan)          # hanya pertanyaan yang dikirim ke model
    hasil.append(dict(id_qa=row.id_qa, split=row.split, tipe=row.tipe, pertanyaan=row.pertanyaan,
                      jawaban_model=r["jawaban"], jumlah_token=r["jumlah_token"],
                      terpotong=r["terpotong"], waktu_detik=r["waktu_detik"],
                      max_new_tokens=MAX_NEW, seed=SEED))
    if len(hasil) % 5 == 0:
        pd.DataFrame(hasil).to_csv(OUT, index=False, encoding="utf-8")
        print(f"  {len(hasil)}/{len(dev)} selesai ({time.time()-t_mulai:.0f} detik)")
pd.DataFrame(hasil).to_csv(OUT, index=False, encoding="utf-8")

d = pd.DataFrame(hasil)
d["jumlah_token"] = d.jumlah_token.astype(int)
d["waktu_detik"] = d.waktu_detik.astype(float)
d["terpotong"] = d.terpotong.astype(str).str.lower().eq("true")

print("\n===== RINGKASAN DEV (batas", MAX_NEW, "token) =====")
print(f"Jumlah jawaban      : {len(d)}")
print(f"Terpotong           : {d.terpotong.sum()} ({d.terpotong.mean()*100:.1f}%)  [target di bawah sekitar 5%]")
print(f"Token  rata/median/maks: {d.jumlah_token.mean():.0f} / {d.jumlah_token.median():.0f} / {d.jumlah_token.max()}")
print(f"Waktu  rata/maks (detik): {d.waktu_detik.mean():.1f} / {d.waktu_detik.max():.1f} | total {d.waktu_detik.sum()/60:.1f} menit")
print(f"Memori GPU puncak   : {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("\nTerpotong per tipe soal:")
print(d.groupby("tipe").terpotong.agg(["sum", "count"]).to_string())

print("\n--- Soal yang terpotong ---")
for _, x in d[d.terpotong].iterrows():
    print(f"{x.id_qa} [{x.tipe}] {x.pertanyaan}  (token={x.jumlah_token})")

print("\n--- 4 contoh jawaban pertama (maks 300 karakter) ---")
for _, x in d.head(4).iterrows():
    print(f"\n{x.id_qa} | {x.pertanyaan}\n> {x.jawaban_model[:300]}")

Batas jawaban: 384 token | soal dev: 31 | sudah selesai: 0
  5/31 selesai (70 detik)
  10/31 selesai (114 detik)
  15/31 selesai (158 detik)
  20/31 selesai (179 detik)
  25/31 selesai (220 detik)
  30/31 selesai (253 detik)

===== RINGKASAN DEV (batas 384 token) =====
Jumlah jawaban      : 31
Terpotong           : 2 (6.5%)  [target di bawah sekitar 5%]
Token  rata/median/maks: 158 / 138 / 384
Waktu  rata/maks (detik): 8.9 / 21.8 | total 4.6 menit
Memori GPU puncak   : 8.17 GB

Terpotong per tipe soal:
          sum  count
tipe                
Biaya       0      1
Durasi      0      5
Instansi    0      3
Prosedur    1      4
Syarat      1     18

--- Soal yang terpotong ---
QA004 [Prosedur] Apa saja tahapan mekanisme pengesahan permohonan paspor untuk anak yang lahir di luar negeri?  (token=384)
QA092 [Syarat] Dokumen apa yang harus diserahkan WNI yang datang dari luar negeri selain formulir F-1.03?  (token=384)

--- 4 contoh jawaban pertama (maks 300 karakter) ---

QA001 | Kepada sia

In [3]:
import time
import pandas as pd
from pathlib import Path

MAX_NEW = 512   # dinaikkan dari 384 (terpotong 6,5% di dev); fungsi jawab() membaca nilai ini

OUT = Path(REPO) / "data" / "processed" / "jawaban_s1_dev_512.csv"
dev = qa[qa.split == "dev"].reset_index(drop=True)
assert len(dev) == 31, f"soal dev {len(dev)}, seharusnya 31"

hasil = pd.read_csv(OUT, dtype=str, keep_default_na=False).to_dict("records") if OUT.exists() else []
selesai = {r["id_qa"] for r in hasil}
print(f"Batas jawaban: {MAX_NEW} token | soal dev: {len(dev)} | sudah selesai: {len(selesai)}")

t_mulai = time.time()
for i, row in dev.iterrows():
    if row.id_qa in selesai:
        continue
    r = jawab(row.pertanyaan)          # hanya pertanyaan yang dikirim ke model
    hasil.append(dict(id_qa=row.id_qa, split=row.split, tipe=row.tipe, pertanyaan=row.pertanyaan,
                      jawaban_model=r["jawaban"], jumlah_token=r["jumlah_token"],
                      terpotong=r["terpotong"], waktu_detik=r["waktu_detik"],
                      max_new_tokens=MAX_NEW, seed=SEED))
    if len(hasil) % 5 == 0:
        pd.DataFrame(hasil).to_csv(OUT, index=False, encoding="utf-8")
        print(f"  {len(hasil)}/{len(dev)} selesai ({time.time()-t_mulai:.0f} detik)")
pd.DataFrame(hasil).to_csv(OUT, index=False, encoding="utf-8")

d = pd.DataFrame(hasil)
d["jumlah_token"] = d.jumlah_token.astype(int)
d["waktu_detik"] = d.waktu_detik.astype(float)
d["terpotong"] = d.terpotong.astype(str).str.lower().eq("true")

print("\n===== RINGKASAN DEV (batas", MAX_NEW, "token) =====")
print(f"Jumlah jawaban      : {len(d)}")
print(f"Terpotong           : {d.terpotong.sum()} ({d.terpotong.mean()*100:.1f}%)  [target di bawah sekitar 5%]")
print(f"Token  rata/median/maks: {d.jumlah_token.mean():.0f} / {d.jumlah_token.median():.0f} / {d.jumlah_token.max()}")
print(f"Waktu  rata/maks (detik): {d.waktu_detik.mean():.1f} / {d.waktu_detik.max():.1f} | total {d.waktu_detik.sum()/60:.1f} menit")
print(f"Memori GPU puncak   : {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("\nTerpotong per tipe soal:")
print(d.groupby("tipe").terpotong.agg(["sum", "count"]).to_string())

print("\n--- Soal yang terpotong ---")
for _, x in d[d.terpotong].iterrows():
    print(f"{x.id_qa} [{x.tipe}] {x.pertanyaan}  (token={x.jumlah_token})")

print("\n--- Jawaban terpanjang: 3 teratas (maks 300 karakter) ---")
for _, x in d.sort_values("jumlah_token", ascending=False).head(3).iterrows():
    print(f"\n{x.id_qa} | {x.pertanyaan} (token={x.jumlah_token})\n> {x.jawaban_model[:300]}")

Batas jawaban: 512 token | soal dev: 31 | sudah selesai: 0
  5/31 selesai (76 detik)
  10/31 selesai (120 detik)
  15/31 selesai (165 detik)
  20/31 selesai (185 detik)
  25/31 selesai (228 detik)
  30/31 selesai (260 detik)

===== RINGKASAN DEV (batas 512 token) =====
Jumlah jawaban      : 31
Terpotong           : 0 (0.0%)  [target di bawah sekitar 5%]
Token  rata/median/maks: 161 / 138 / 472
Waktu  rata/maks (detik): 9.1 / 27.0 | total 4.7 menit
Memori GPU puncak   : 8.17 GB

Terpotong per tipe soal:
          sum  count
tipe                
Biaya       0      1
Durasi      0      5
Instansi    0      3
Prosedur    0      4
Syarat      0     18

--- Soal yang terpotong ---

--- Jawaban terpanjang: 3 teratas (maks 300 karakter) ---

QA004 | Apa saja tahapan mekanisme pengesahan permohonan paspor untuk anak yang lahir di luar negeri? (token=472)
> Tahapan mekanisme pengesahan permohonan paspor untuk anak yang lahir di luar negeri sebagai berikut:

1. **Pengajuan Permohonan** – Orang tu

penguncian

In [10]:
import json, zipfile, datetime, platform
from pathlib import Path
import pandas as pd, torch, transformers
from google.colab import files

R = Path(REPO); P = R / "data" / "processed"
print("Isi data/processed:")
for f in sorted(P.iterdir()):
    print(f"  {f.name:35s} {f.stat().st_size:>10,d} byte")

# baca semua berkas dev; batas token diambil dari isi berkas
ada, ringkasan = {}, {}
for p in sorted(P.glob("jawaban_s1_dev*.csv")):
    d = pd.read_csv(p, dtype=str, keep_default_na=False)
    batas = int(d.max_new_tokens.iloc[0])
    ada[batas] = p
    t = d.terpotong.str.lower().eq("true").sum()
    ringkasan[str(batas)] = dict(jumlah=len(d), terpotong=int(t), persen=round(t/len(d)*100, 1),
        token_rata2=round(d.jumlah_token.astype(int).mean(), 1),
        waktu_rata2_detik=round(d.waktu_detik.astype(float).mean(), 1),
        berkas=p.name, sumber="berkas hasil")
print("\nBatas token yang ditemukan di berkas:", sorted(ada))

# hasil lama yang berkasnya tidak tersimpan: dicatat dari keluaran layar sel 5.3
cadangan = {"256": dict(jumlah=31, terpotong=7, persen=22.6, token_rata2=145, waktu_rata2_detik=7.9),
            "384": dict(jumlah=31, terpotong=2, persen=6.5, token_rata2=158, waktu_rata2_detik=8.6)}
for k, v in cadangan.items():
    if k not in ringkasan:
        ringkasan[k] = dict(v, berkas="tidak tersimpan", sumber="keluaran layar sel 5.3")

assert 512 in ada, "hasil dev 512 token TIDAK ADA: jangan lanjut, kirim daftar berkas di atas ke saya"
assert ringkasan["512"]["jumlah"] == 31 and ringkasan["512"]["terpotong"] == 0, "hasil 512 tidak sesuai (31 baris, 0 terpotong)"

config = dict(
    tanggal_dikunci=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).strftime("%Y-%m-%d %H:%M WIB"),
    skenario="S1 (tanpa RAG)", model=MODEL_ID, revisi=REVISI, dtype="float16",
    decoding="greedy (do_sample=False)", max_new_tokens=512, seed=SEED, system_prompt=SYSTEM,
    masukan_ke_model="hanya kolom pertanyaan; jawaban_acuan dan bukti_kutipan tidak dikirim",
    data=dict(file="data/processed/qa.csv", jumlah_qa=int(len(qa)),
              dev=int((qa.split == "dev").sum()), uji=int((qa.split == "uji").sum())),
    pemilihan_max_new_tokens=dict(dasar="hanya soal dev (31 soal)", hasil_dev=ringkasan,
                                  keputusan="512 token (0% terpotong di dev)"),
    lingkungan=dict(gpu=torch.cuda.get_device_name(0), torch=torch.__version__,
                    transformers=transformers.__version__, python=platform.python_version()),
    catatan="Konfigurasi dikunci SEBELUM 63 soal uji dijalankan.")

(R / "docs").mkdir(exist_ok=True)
cfg = R / "docs" / "config_s1.json"
cfg.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

zp = Path("s1_dev_dan_config.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(cfg, "docs/config_s1.json")
    for p in ada.values():
        z.write(p, f"data/processed/{p.name}")
print("\nIsi zip:", zipfile.ZipFile(zp).namelist())
print(json.dumps(config["pemilihan_max_new_tokens"], indent=2, ensure_ascii=False))
files.download(str(zp))

Isi data/processed:
  .gitkeep                                     0 byte
  chunks.csv                             102,800 byte
  jawaban_s1_dev.csv                      20,021 byte
  jawaban_s1_dev_384.csv                  19,702 byte
  jawaban_s1_dev_512.csv                  20,024 byte
  qa.csv                                  30,619 byte

Batas token yang ditemukan di berkas: [384, 512]

Isi zip: ['docs/config_s1.json', 'data/processed/jawaban_s1_dev_512.csv', 'data/processed/jawaban_s1_dev_384.csv']
{
  "dasar": "hanya soal dev (31 soal)",
  "hasil_dev": {
    "512": {
      "jumlah": 31,
      "terpotong": 0,
      "persen": 0.0,
      "token_rata2": 160.8,
      "waktu_rata2_detik": 9.1,
      "berkas": "jawaban_s1_dev_512.csv",
      "sumber": "berkas hasil"
    },
    "384": {
      "jumlah": 31,
      "terpotong": 2,
      "persen": 6.5,
      "token_rata2": 157.7,
      "waktu_rata2_detik": 8.9,
      "berkas": "jawaban_s1_dev_384.csv",
      "sumber": "berkas hasil"
    },


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>